# Reproducing Figure 3: Alignment Scaling with Frames and Captions

This notebook demonstrates how to reproduce the alignment scaling analysis
from Figure 3 of the VPRH paper. It computes pairwise alignment scores between
vision and language model features as a function of the number of video frames
and text captions, and fits a parametric scaling law.

**⚠️ Disclaimer:** This file is for reference purposes only. The raw feature files used in the paper are not included in this release.

In the paper we showed this for the: 

1. Dataset: Vatex dataset
2. Video Models: VideoMAEv2-Huge, DINOv2 Giant
3. LLM Model: Gemma 2-9b-it

and that this script assumes that all of the features have been precomputed.

In [ ]:
# Basic Imports
import os
import re

import numpy as np
import torch

from vprh import alignment

In [ ]:
# Helper functions for loading pre-extracted features.

def load_gemma_feature(filename):
    """Load a Gemma (language model) feature file."""
    feature_name = os.path.basename(filename).replace(
        'gemma_stacked_', '').replace('.npy', '')
    global_var_name = f'{feature_name}_gemma_features'

    # Check if features are already loaded
    if global_var_name in globals():
        print(f"{feature_name} features already loaded.")
        features = globals()[global_var_name]
    else:
        print(f"Loading {global_var_name}...")
        features = np.load(filename)
        globals()[global_var_name] = features

    print(f"{feature_name} features size: ", features.shape)
    return (feature_name, features)


def load_feature(filename):
    """Load a vision model feature file."""
    feature_name = os.path.basename(filename).replace(
        'extracted_features_', '').replace('.npy', '')
    global_var_name = f'{feature_name}_features'

    # Check if features are already loaded
    if global_var_name in globals():
        print(f"{feature_name} features already loaded.")
        features = globals()[global_var_name]
    else:
        print(f"Loading {global_var_name}...")
        features = np.load(filename)
        globals()[global_var_name] = features

    print(f"{feature_name} features size: ", features.shape)
    return (feature_name, features)


def find_Xcaptions_features(path):
    """Find all Gemma caption feature files matching *_Xcaptions_feats.npy."""
    regex = re.compile(r'_(\d+)captions_feats\.npy$')
    all_features_dict = {}

    for dirname, _, filenames in os.walk(path):
        for filename in filenames:
            if filename.startswith('gemma_stacked_') and regex.search(filename):
                filepath = os.path.join(dirname, filename)
                feature_name, features = load_gemma_feature(filepath)
                all_features_dict[feature_name] = torch.from_numpy(features)

    return all_features_dict


def find_Xframes_features(path):
    """Find all vision feature files matching *_Xframes.npy."""
    regex = re.compile(r'_(\d+)frames\.npy$')
    all_features_dict = {}

    for dirname, _, filenames in os.walk(path):
        for filename in filenames:
            if filename.startswith('extracted_features_') and regex.search(filename):
                filepath = os.path.join(dirname, filename)
                feature_name, features = load_feature(filepath)
                all_features_dict[feature_name] = torch.from_numpy(features)

    return all_features_dict

## Set Feature Paths

**Edit the paths below** to point to the directory where you have extracted
your features using the provided scripts. The expected directory structure is:

```
your_feature_dir/
├── gemma_stacked_*_Xcaptions_feats.npy   # Language model features
└── vatex_data_multiframe/
    └── extracted_features_*_Xframes.npy  # Vision model features
```

In [ ]:
# ============================================================
# EDIT THIS PATH to point to your local feature directory.
# ============================================================
LOCAL_FEATURE_DIR = '/path/to/your/features/'

## Compute Alignment Scores

Load the pre-extracted caption and frame features and compute pairwise
alignment scores using the mutual k-NN metric.

In [ ]:
# Compute the alignment scores
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

feats_dict = find_Xcaptions_features(LOCAL_FEATURE_DIR)

# Group scores by model name
alignments = {}

vision_features_dir = os.path.join(LOCAL_FEATURE_DIR, 'vatex_data_multiframe/')
print(f"Looking for vision features in: {vision_features_dir}")

for text_feats_name, text_feats in feats_dict.items():
    # Extract model name and number of captions
    match = re.match(r'(\d+)captions_feats\.npy$', text_feats_name + '.npy')
    if match:
        model_name = 'gemma'
        num_captions = int(match.group(1))
        print(f'Found {model_name} with {num_captions} captions')
    else:
        print(f"Skipping file with unexpected name format: {text_feats_name}")
        continue

    test_vision_features = [
        'extracted_features_imagenet_dinov2_1frames.npy',
        'extracted_features_imagenet_dinov2_2frames.npy',
        'extracted_features_imagenet_dinov2_4frames.npy',
        'extracted_features_imagenet_dinov2_8frames.npy',
        'extracted_features_imagenet_dinov2_10frames.npy',
        'extracted_features_imagenet_dinov2_12frames.npy',
        'extracted_features_imagenet_dinov2_16frames.npy',
    ]

    for vision_feature in test_vision_features:
        filename = os.path.join(vision_features_dir, vision_feature)
        vision_feats_name, vision_feats = load_feature(filename)
        features1 = alignment.prepare_features(text_feats.float())
        features2 = alignment.prepare_features(
            torch.from_numpy(vision_feats).float())

        # Compute alignment score
        if vision_feats_name not in alignments:
            alignments[vision_feats_name] = {}
        if num_captions not in alignments[vision_feats_name]:
            best_score, _, _, _, _ = alignment.compute_score(
                features1, features2, normalize=True)
            print(f"{vision_feats_name} vs {text_feats_name} score = {best_score}")
            alignments[vision_feats_name][num_captions] = best_score

## Plot: Alignment vs. Number of Captions

This plot shows how alignment score changes as we vary the number of text
captions used for the language model features, for a fixed vision model
(DINOv2) at different frame counts.

In [ ]:
# Plot the alignment as a function of number of captions
plt.figure(figsize=(12, 6))

# Define line and marker styles
line_styles = {'simple_mae_vit_v0': '-'}
marker_styles = {'simple_mae_vit_v0': 'o'}

# Assign colors based on number of frames
colors = {}
for vision_feature_name in alignments.keys():
    parts = vision_feature_name.split('_')
    num_frames = int(parts[-1].replace('frames', ''))
    if num_frames not in colors:
        colors[num_frames] = f'C{len(colors)}'

for vision_feature_name, scores_by_captions in alignments.items():
    sorted_captions = sorted(scores_by_captions.keys())
    sorted_scores = [scores_by_captions[nc] for nc in sorted_captions]

    parts = vision_feature_name.split('_')
    vision_model_name = '_'.join(parts[:-1])
    num_frames = int(parts[-1].replace('frames', ''))
    label_name = f'DINOv2 {num_frames} frames'

    plt.plot(sorted_captions, sorted_scores,
             marker=marker_styles.get(vision_model_name, 'o'),
             linestyle=line_styles.get(vision_model_name, '-'),
             color=colors[num_frames], label=label_name)

plt.xlabel('Number of Captions', fontsize=22)
plt.ylabel('Alignment against Gemma-2-9b-it', fontsize=18)
plt.legend(fontsize=12)
plt.grid(True, which="both", ls="--")
plt.xscale('log')
plt.yscale('log')


def format_log_decimal(y, pos):
    return f'{y:.2f}'


plt.gca().yaxis.set_minor_formatter(ticker.FuncFormatter(format_log_decimal))

# Place ticks at integer positions 1-10
all_integer_ticks = [i for i in range(1, 11)]
plt.gca().xaxis.set_major_locator(ticker.FixedLocator(all_integer_ticks))

x_formatter = ticker.ScalarFormatter(useOffset=False, useMathText=False)
x_formatter.set_scientific(False)
plt.gca().xaxis.set_major_formatter(x_formatter)
plt.gca().xaxis.set_minor_locator(
    ticker.LogLocator(base=10.0, subs='all'))

plt.savefig('alignment_vs_captions.pdf')
plt.show()

## Plot: Alignment vs. Number of Frames

This plot restructures the data to show alignment as a function of the number
of video frames, with separate curves for different caption counts.

In [ ]:
# Restructure the data and plot alignment as a function of the number of frames
from collections import defaultdict

# We plot a subset of curves to avoid clutter.
test_num_captions = [1, 2, 4, 8, 10]

# Restructure: group scores by (model, captions) -> {frames: score}
model_data = defaultdict(lambda: defaultdict(list))
for vision_feature_name, scores_by_captions in alignments.items():
    parts = vision_feature_name.split('_')
    vision_model_name = '_'.join(parts[:-1])
    num_frames = int(parts[-1].replace('frames', ''))
    for num_captions, score in scores_by_captions.items():
        if num_captions not in test_num_captions:
            continue
        combined_model_name = f"{vision_model_name}_{num_captions}captions"
        model_data[combined_model_name][num_frames].append(score)


def sort_key(model_string):
    """Parse 'model_Xcaptions' into (model_name, num_captions) for sorting."""
    parts_no_captions = model_string.rsplit('_', 1)
    model_part = parts_no_captions[0]
    caption_part_int = int(parts_no_captions[1].replace('captions', ''))
    return (model_part, caption_part_int)


sorted_keys = sorted(model_data.keys(), key=sort_key)
model_data = {key: model_data[key] for key in sorted_keys}

# Assign colors based on unique caption counts
line_styles = {'simple_mae_vit_v0': '-'}
marker_styles = {'simple_mae_vit_v0': 'o'}

unique_captions = sorted(list(set(
    int(re.search(r'\d+', k.split('_')[-1]).group(0))
    for k in model_data.keys()
)))
print(unique_captions)
colors = {caption: f'C{i}' for i, caption in enumerate(unique_captions)}

# Plot
plt.figure(figsize=(12, 6))
for model_name, scores_by_frames in model_data.items():
    sorted_frames = sorted(scores_by_frames.keys())
    sorted_scores = [scores_by_frames[nf] for nf in sorted_frames]

    parts = model_name.split('_')
    vision_model_name = '_'.join(parts[:-1])
    num_captions = int(re.search(r'\d+', parts[-1]).group(0))
    label_name = f'DINOv2 {num_captions} captions'

    plt.plot(sorted_frames, sorted_scores,
             marker=marker_styles.get(vision_model_name, 'o'),
             linestyle=line_styles.get(vision_model_name, '-'),
             color=colors[num_captions], label=label_name)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of Frames', fontsize=22)
plt.ylabel('Alignment against Gemma-2-9b-it', fontsize=18)
plt.legend(fontsize=12)
plt.grid(True, which="both", ls="--")

plt.gca().yaxis.set_minor_formatter(
    ticker.FuncFormatter(format_log_decimal))

all_integer_ticks = (1, 2, 4, 8, 16)
plt.gca().xaxis.set_major_locator(ticker.FixedLocator(all_integer_ticks))

x_formatter = ticker.ScalarFormatter(useOffset=False, useMathText=False)
x_formatter.set_scientific(False)
plt.gca().xaxis.set_major_formatter(x_formatter)

plt.savefig('alignment_vs_frames.pdf')
plt.show()

## Fit Parametric Scaling Law

Fit the saturation model from the paper:

$$\text{score} = S_{\max} - \left(A \cdot \text{frames}^{-\alpha} + B \cdot \text{captions}^{-\beta}\right)$$

This model captures how alignment saturates as both data axes (frames and
captions) increase.

In [ ]:
# Fit the parametric model
from scipy.optimize import curve_fit

# Restructure the data to group by (model, frames, captions)
model_data_fit = defaultdict(lambda: defaultdict(dict))
for vision_feature_name, scores_by_captions in alignments.items():
    parts = vision_feature_name.split('_')
    vision_model_name = '_'.join(parts[:-1])
    num_frames = int(parts[-1].replace('frames', ''))
    for num_captions, score in scores_by_captions.items():
        model_data_fit[vision_model_name][num_frames][num_captions] = score


def saturation_model(X, S_max, A, alpha, B, beta):
    """Parametric saturation model for alignment scaling."""
    frames, captions = X
    safe_frames = np.maximum(frames, 1e-9)
    safe_captions = np.maximum(captions, 1e-9)
    return S_max - (A * safe_frames**(-alpha) + B * safe_captions**(-beta))


for model_name, data in model_data_fit.items():
    print(f"--- Fitting saturation model for: {model_name} ---")

    # Flatten the nested dict into parallel arrays
    frames_list, captions_list, scores_list = [], [], []
    for frames, captions_scores in data.items():
        for captions, score in captions_scores.items():
            frames_list.append(frames)
            captions_list.append(captions)
            scores_list.append(score)

    frames_arr = np.array(frames_list, dtype=np.float64)
    captions_arr = np.array(captions_list, dtype=np.float64)
    scores_arr = np.array(scores_list, dtype=np.float64)
    X_data = np.vstack((frames_arr, captions_arr))

    # Initial guesses for [S_max, A, alpha, B, beta]
    initial_guess = [np.max(scores_arr), 1.0, 0.5, 1.0, 0.5]
    lower_bounds = [0, 0, 0, 0, 0]
    upper_bounds = [np.inf, np.inf, np.inf, np.inf, np.inf]

    try:
        params, covariance = curve_fit(
            saturation_model,
            X_data,
            scores_arr,
            p0=initial_guess,
            bounds=(lower_bounds, upper_bounds),
            maxfev=5000,
        )

        S_max_fit, A_fit, alpha_fit, B_fit, beta_fit = params

        # Evaluate goodness of fit
        scores_predicted = saturation_model(X_data, *params)
        ss_res = np.sum((scores_arr - scores_predicted) ** 2)
        ss_tot = np.sum((scores_arr - np.mean(scores_arr)) ** 2)
        r_squared = 1 - (ss_res / ss_tot)

        print(f"R-squared: {r_squared:.4f}")
        print(
            f"Fitted law: score = {S_max_fit:.2f}"
            f" - ({A_fit:.2f}*frames^-{alpha_fit:.2f}"
            f" + {B_fit:.2f}*captions^-{beta_fit:.2f})"
        )
        print()

    except RuntimeError:
        print(
            "Fit failed for this model."
            " Consider adjusting initial guesses or bounds."
        )